# Installation check

_Copyright (C) 2024-, J.T. Lizier; Distributed under GNU General Public License v3_

This is a sample notebook to run to check that your installation works ok

**Step 1**: Check standard libraries

In [ ]:
import numpy as np
import scipy
import matplotlib
import pandas
import torch
# The following should all be built-in already:
import random
import os
import math
import string
import re

print("✅ all required packages installed.")

**Step 2**: Check that the platform and python installation are match on 64-bits (else match on 32 bits)

In [ ]:
import platform
import struct

# Synthesising suggestions from Copilot and ChatGPT:
# a. Machine architecture (OS/CPU)
#    Don't use platform.architecture() as this queries the Python executable.
machine_arch = platform.machine()
#    Now parse this to 32 or 64 bits
#    platform.machine() gives strings like 'x86_64', 'AMD64', 'i386', 'arm64', etc.
#    May need to update this in future:
if "64" in machine_arch:
    os_arch = 64
elif "86" in machine_arch or "32" in machine_arch:
    os_arch = 32
else:
    os_arch = "Unknown"
# b. Python interpreter architecture
python_arch = struct.calcsize("P") * 8

print(f"Machine: {os_arch}-bit, Python: {python_arch}-bit")
if (os_arch == python_arch):
    print("✅ Machine and Python architectures match")
else:
    print("❌ Machine and Python architectures do not match!")

**Step 3**: Check that jpype1 is installed:

In [ ]:
try:
    from jpype import *
    print("✅ jpype1 is already installed.")
except ImportError:
    print("❌ jpype1 is not installed!")
    # In principle, we could try to install jpype1 for the user as:
    #  !pip install --user jpype1
    # However there are too many complications (e.g. pip-v-pip3, externally managed environments, etc).
    # So we will ask the user to install jpype1 themselves
    print("Please use your python package manager (e.g. pip, pip3, homebrew, conda, etc) to install *jpype1* (not jpype).")
    print("Then restart the kernel for this notebook, and run the notebook again.")

**Step 4**: Check that the `infodynamics.jar` is in the expected location:

In [ ]:
# Locate the JIDT jar library -- it should be two folders up from the location of this notebook.
jarLocation = os.path.join(os.getcwd(), "..", "..", "infodynamics.jar");
if (not(os.path.isfile(jarLocation))):
    raise Exception("infodynamics.jar not found (expected at " + os.path.abspath(jarLocation))
else:
    print("✅ infodynamics.jar is in the expected location.")

**Step 5**: Check that the Java Virtual Machine (JVM) can be started by the python notebook:

In [ ]:
if (not isJVMStarted()):
    # Add JIDT jar library to the path and
    # Start the JVM (add the "-Xmx" option with say 1024M if you get crashes due to not enough memory space)
    # This should raise an Exception if unsuccessful
    startJVM(getDefaultJVMPath(), "-ea", "-Djava.class.path=" + jarLocation)
    if (isJVMStarted()):
        print("✅ JVM started")
    else:
        raise Exception("❌ startJVM() ran without exception but the JVM is not started ...?")
else:
    print("✅ JVM was already started")

**Step 6**: Run a simple calculation with JIDT:

In [ ]:
# Generate some random binary data for a simple calculation.
# Here destArray is a lagged copy of sourceArray:
sourceArray = np.random.randint(0, 2, size=100)
destArray   = np.empty(100, dtype=int)
destArray[0] = 0
destArray[1:] = sourceArray[:99]

# Create a TE calculator and run it:
teCalcClass = JPackage("infodynamics.measures.discrete").TransferEntropyCalculatorDiscrete
teCalc = teCalcClass()
teCalc.initialise()
teCalc.addObservations(sourceArray, destArray)
result = teCalc.computeAverageLocalOfObservations()
print("✅ Calculation ran ok, check that it returned a value close to 1 bit : %.4f bits" % result)